# 0. Prepare Modeling Data

This notebook loads the scaled fraud dataset, splits it into train and test sets, applies **SMOTENC** (Experiment 3 strategy: `sampling_strategy=0.1`) only to the training split, and saves model-ready artifacts (X_train, X_test, y_train, y_test as NPZ, plus a features JSON) for downstream training notebooks.

## 1. Imports & Setup

Set up the notebook environment, file paths, and modeling constants.

In [1]:
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTENC
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Detect project root by searching up for requirements.txt / README.md
BASE_DIR = None
for candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (candidate / 'dataset').exists() or (candidate / 'requirements.txt').exists() or (candidate / 'README.md').exists():
        BASE_DIR = candidate
        break
if BASE_DIR is None:
    BASE_DIR = Path.cwd()
    print('Warning: repository root not found; using CWD as BASE_DIR')

INPUT_PATH  = BASE_DIR / 'dataset' / 'processed' / 'credit_card_fraud_scaled.csv'
OUTPUT_DIR  = BASE_DIR / 'artifacts' / 'data'
EVAL_DIR    = BASE_DIR / 'artifacts' / 'evaluation'
EVAL_DIR.mkdir(parents=True, exist_ok=True)
TARGET_COL  = 'is_fraud'

# Continuous columns that were StandardScaled — everything else is categorical for SMOTENC
CONTINUOUS_COLS = [
    'amount', 'amount_log',
    'velocity_last_24h', 'velocity_last_24h_log',
    'city_population', 'city_population_log',
]

# SMOTENC sampling strategy (Experiment 3)
SMOTENC_SAMPLING_STRATEGY = 0.1
# Optimal classification threshold found in Experiment 3
OPTIMAL_THRESHOLD = 0.70

print(f'Imports and setup complete. BASE_DIR={BASE_DIR}')

Imports and setup complete. BASE_DIR=C:\Users\2021ICTS28\Desktop\end-to-end-credit-card-fraud-detection-system


## 2. Load Scaled Dataset

Read the scaled input file. No identifier column is present in this dataset.

In [2]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Input file not found: {INPUT_PATH}. Run 4_handle_scaling.ipynb first.')

df = pd.read_csv(INPUT_PATH)
print(f'Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]:,} columns')

if TARGET_COL not in df.columns:
    raise ValueError(f'Missing target column: {TARGET_COL}')

print('\nColumn list:')
print(list(df.columns))
df.head()

Loaded dataset: 1,296,675 rows x 30 columns

Column list:
['day_of_week', 'is_weekend', 'location_mismatch', 'velocity_last_24h', 'foreign_transaction', 'amount', 'city_population', 'is_fraud', 'amount_log', 'velocity_last_24h_log', 'city_population_log', 'customer_age_binned', 'transaction_hour_binned', 'distance_to_merchant_binned', 'merchant_category_entertainment', 'merchant_category_food_dining', 'merchant_category_gas_transport', 'merchant_category_grocery_net', 'merchant_category_grocery_pos', 'merchant_category_health_fitness', 'merchant_category_home', 'merchant_category_kids_pets', 'merchant_category_misc_net', 'merchant_category_misc_pos', 'merchant_category_personal_care', 'merchant_category_shopping_net', 'merchant_category_shopping_pos', 'merchant_category_travel', 'gender_F', 'gender_M']


## 3. Stratified Train / Test Split

Separate features from the target and create an 80/20 stratified split.

In [3]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Target distribution in full dataset:')
print(y.value_counts().sort_index())
print(f'\nTrain split: {X_train.shape[0]:,} rows x {X_train.shape[1]:,} columns')
print(f'Test  split: {X_test.shape[0]:,} rows x {X_test.shape[1]:,} columns')
print('\nTrain class distribution:')
print(y_train.value_counts().sort_index())

Target distribution in full dataset:
is_fraud
0    1289169
1       7506
Name: count, dtype: int64

Train split: 1,037,340 rows x 29 columns
Test  split: 259,335 rows x 29 columns

Train class distribution:
is_fraud
0    1031335
1       6005
Name: count, dtype: int64


## 4. Identify Categorical Features for SMOTENC

Everything except the 6 continuous columns is treated as categorical for SMOTENC.

In [4]:
categorical_cols     = [col for col in X_train.columns if col not in CONTINUOUS_COLS]
categorical_features = [X_train.columns.get_loc(col) for col in categorical_cols]

if not categorical_features:
    raise ValueError('No categorical columns were detected for SMOTENC.')

print(f'Continuous columns  ({len(CONTINUOUS_COLS)}): {CONTINUOUS_COLS}')
print(f'Categorical columns ({len(categorical_cols)}): {categorical_cols}')

Continuous columns  (6): ['amount', 'amount_log', 'velocity_last_24h', 'velocity_last_24h_log', 'city_population', 'city_population_log']
Categorical columns (23): ['day_of_week', 'is_weekend', 'location_mismatch', 'foreign_transaction', 'customer_age_binned', 'transaction_hour_binned', 'distance_to_merchant_binned', 'merchant_category_entertainment', 'merchant_category_food_dining', 'merchant_category_gas_transport', 'merchant_category_grocery_net', 'merchant_category_grocery_pos', 'merchant_category_health_fitness', 'merchant_category_home', 'merchant_category_kids_pets', 'merchant_category_misc_net', 'merchant_category_misc_pos', 'merchant_category_personal_care', 'merchant_category_shopping_net', 'merchant_category_shopping_pos', 'merchant_category_travel', 'gender_F', 'gender_M']


## 5. Apply SMOTENC (Experiment 3 — sampling_strategy=0.1)

Oversample **only** the training split to avoid data leakage into the test set.  
This corresponds to **Experiment 3**: SMOTENC + threshold tuning at **0.70**, which achieved the best F1-score of **0.8138** (Precision: 0.82, Recall: 0.80, ROC-AUC: 0.9980).

In [5]:
print(f'Applying SMOTENC with sampling_strategy={SMOTENC_SAMPLING_STRATEGY}...')
smotenc = SMOTENC(
    categorical_features=categorical_features,
    random_state=42,
    sampling_strategy=SMOTENC_SAMPLING_STRATEGY
)
X_train_resampled, y_train_resampled = smotenc.fit_resample(X_train, y_train)

print('Target distribution after SMOTENC (training set only):')
print(pd.Series(y_train_resampled).value_counts().sort_index())
print(f'\nResampled train shape: {X_train_resampled.shape[0]:,} rows x {X_train_resampled.shape[1]:,} columns')

Applying SMOTENC with sampling_strategy=0.1...
Target distribution after SMOTENC (training set only):
is_fraud
0    1031335
1     103133
Name: count, dtype: int64

Resampled train shape: 1,134,468 rows x 29 columns


## 6. Visualize Class Balance

Compare class distribution before and after resampling.

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

pd.Series(y_train).value_counts().sort_index().plot(
    kind='bar', ax=axes[0], title='Training Before SMOTENC', color=['steelblue', 'tomato'])
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')

pd.Series(y_train_resampled).value_counts().sort_index().plot(
    kind='bar', ax=axes[1], title='Training After SMOTENC', color=['steelblue', 'tomato'])
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')

pd.Series(y_test).value_counts().sort_index().plot(
    kind='bar', ax=axes[2], title='Test Set (untouched)', color=['steelblue', 'tomato'])
axes[2].set_xlabel('Class')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig(EVAL_DIR / 'class_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Class balance chart saved.')

Class balance chart saved.


## 7. Save Artifacts

Save the resampled training split + original test split as compressed NPZ files,  
plus a `features.json` listing column names. These are consumed by `1_base_model_training.ipynb`.

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.savez_compressed(OUTPUT_DIR / 'credit_card_fraud_X_train.npz', X_train=X_train_resampled.values)
np.savez_compressed(OUTPUT_DIR / 'credit_card_fraud_X_test.npz',  X_test=X_test.values)
np.savez_compressed(OUTPUT_DIR / 'credit_card_fraud_y_train.npz', y_train=y_train_resampled.values)
np.savez_compressed(OUTPUT_DIR / 'credit_card_fraud_y_test.npz',  y_test=y_test.values)

feature_names = list(X_train_resampled.columns)
with open(OUTPUT_DIR / 'features.json', 'w') as f:
    json.dump(feature_names, f, indent=2)

# Save the optimal threshold for inference
threshold_info = {'optimal_threshold': OPTIMAL_THRESHOLD}
with open(OUTPUT_DIR / 'threshold.json', 'w') as f:
    json.dump(threshold_info, f, indent=2)

print(f'Saved artifacts to: {OUTPUT_DIR}')
print(f'  X_train shape : {X_train_resampled.shape}')
print(f'  X_test  shape : {X_test.shape}')
print(f'  Features saved: {len(feature_names)}')
print(f'  Optimal threshold saved: {OPTIMAL_THRESHOLD}')

Saved artifacts to: C:\Users\2021ICTS28\Desktop\end-to-end-credit-card-fraud-detection-system\artifacts\data
  X_train shape : (1134468, 29)
  X_test  shape : (259335, 29)
  Features saved: 29
  Optimal threshold saved: 0.7
